In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold
from scipy.stats import ttest_ind
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample
from sklearn.linear_model import LassoCV
from sklearn.feature_selection import RFECV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
import pickle
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from scipy.stats import norm

## Initial Preprocessing

In [ ]:
#reading in final dataset from previous notebook
data = pd.read_csv(r"final_data.csv")


In [357]:
#Getting number of columns that contain NaN values
print(len(data.loc[(data['fulfillment_labor_materials'].isna()) | (data['item_quantity'].isna())  | (data['students_reached'].isna()) | (data['item_unit_price'].isna())]))

3985


In [299]:
#Imputing missing values with 0
data[['item_unit_price', 'item_quantity', 'fulfillment_labor_materials', 'students_reached']] = data[['item_unit_price', 'item_quantity', 'fulfillment_labor_materials', 'students_reached']].fillna(0)

In [ ]:
#Defining X and Y
X = data.drop(columns = ['is_exciting', "Unnamed: 0", "projectid"])
y = data['is_exciting']

In [ ]:
#Going to filter out data with very low variance (<.01)
var_df = pd.DataFrame(X.var(), columns = ['variance']).reset_index()
low_var = var_df[var_df['variance'] <= .01]
X = X.drop(columns = list(low_var['index']))

In [302]:
#Checking for highly correlated features and dropping those with a perfect correlation to another feature
corr_matrix = X.corr().abs() 
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_df = upper.stack().reset_index()
high_corr_df.columns = ['Feature1', 'Feature2', 'Correlation']
high_corr_df = high_corr_df[high_corr_df['Correlation'] > 0.8]
#Several features have perfect correlation, we will ax Feature 1
perfect_corr = high_corr_df[high_corr_df['Correlation']  == 1]
X = X.drop (columns = list(perfect_corr['Feature1'])+ ['donation_to_project', 'total_price_excluding_optional_support'])

In [ ]:
#Train test split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = .2, random_state = 7604, stratify = y)

In [ ]:
#Performing a t-test to only get variables where the means are significantly different between the two classifications

results = [] 
class0 = y == 0 
class1 = y == 1 
for col in X.columns: 
    t_stat, p_value = ttest_ind(X.loc[class0, col], X.loc[class1, col], equal_var=False) 
    results.append([col, t_stat, p_value]) 
results_df = pd.DataFrame(results, columns=["feature", "t_stat", "p_value"]) 

#filtering to get variables that have a p - value greater than .05 to drop
results_df = results_df[results_df['p_value'] > .05]
X_train = X_train.drop(columns = list(results_df['feature']))

#Make sure the test data columns get filtered out 
X_test = X_test[list(X_train.columns)]


Doing some EDA on the Training Data

In [ ]:
#Getting Boolean and Float columns for count comparison
X_full = pd.concat((X_train, X_test))
bool_col_names = X_full .columns[X_train.apply(lambda col: set(col.dropna().unique()).issubset({0,1}))]
print(len(bool_col_names))


float_col_names = X_full .select_dtypes(include='float').columns
print(len(float_col_names))

57

In [ ]:
#Get boolean columns with a mean of less than 10%
bool_means = X_train[bool_col_names].mean()
cols_10pct = bool_means[bool_means <= 0.10].index.tolist()
lower_var = len(cols_10pct)
print(lower_var)

74

In [ ]:
higher_var = len(bool_col_names)- len(cols_10pct)
print(higher_var)

32

In [ ]:
#Creating a bar plot to show the difference between the high and low mean columns
plt.figure(figsize= (10,6))
plt.bar(x= ["Higher Mean Columns (>.1)", "Lower Mean Columns (<.1)"], height = [higher_var, lower_var])
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.title("Comparison of Boolean Column Means")
plt.savefig("MeanComparison")

In [ ]:
#Checking out NA data
len_na = len(data.loc[(data['fulfillment_labor_materials'].isna()) | (data['item_quantity'].isna())  | (data['students_reached'].isna()) | (data['item_unit_price'].isna())])
print(len_na)

In [ ]:
#Comparing the number of columns with and without missing columns

plt.figure(figsize= (10,6))
plt.bar(x= ["Number of columns with missing values", "Number of columns with no missing Values"], height = [len_na, 100000-len_na], color = "mediumseagreen")
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.title("Comparison of Columns With and Without Missing Values")
plt.savefig("Na to non-na")

Scaling X data to get variance of continuous columns


In [314]:
#Scale x data 
ss = StandardScaler()
X_train_scaled =ss.fit_transform(X_train)
X_test_scaled= ss.transform(X_test)

In [315]:
X_train_scaled = pd.DataFrame(X_train_scaled, columns = X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns = X_test.columns)

In [ ]:
#Getting top ten continuous columns by variance and plotting
X_scaled_full = pd.concat((X_train_scaled, X_test_scaled))
top10 = (
    X_scaled_full[list(float_col_names)].var()
      .sort_values(ascending=False)
      .head(10)
      .reset_index()
)

top10.columns = ["column", "variance"]

plt.figure(figsize = (15,15))
plt.bar(x = top10['column'], height = top10['variance'], color = "lightcoral", width= .5)
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xticks(range(len(top10)))
ax.set_xticklabels(top10['column'], rotation=45, ha='right', size = 15)
plt.title("Bar Chart Showing Top 10 Continuous Variables by Variance", size = 24)
plt.tight_layout()
plt.savefig("top10var")


In [ ]:

#Plotting histograms and box plots to show distributions
fig, axes = plt.subplots(2, 5, figsize=(20, 8))


axes = axes.flatten()

for i, ax in enumerate(axes):
    ax.hist(X_scaled_full[list(top10["column"])[i]])
    ax.set_title("Histogram for Variable " + list(top10["column"])[i])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


plt.tight_layout()
plt.savefig("Histograms")
    
fig, axes = plt.subplots(2, 5, figsize=(20, 8))


axes = axes.flatten()

for i, ax in enumerate(axes):
    ax.boxplot(X_scaled_full[list(top10["column"])[i]])
    ax.set_title("Boxplot for Variable " + list(top10["column"])[i])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


plt.tight_layout()
plt.savefig("Boxplots")


Stability Selection with Lasso Penalty

In [126]:
n_subsamples = 100
subsample_fractions = [0.3, 0.5, 0.7, 0.9]
thresholds = [.3, 0.5, 0.7, 0.9]

results = {}

for frac in subsample_fractions:
    selection_counts = np.zeros(len(X_train_scaled.columns))

    for _ in range(n_subsamples):
        if _ % 10 == 0:
            print(f"  Progress: {_}/{n_subsamples}")

        X_sub, y_sub = resample(X_train_scaled, y_train, n_samples=int(80000 * frac))

        model = LassoCV(cv=5, n_jobs=-1).fit(X_sub, y_sub)
        
        selection_counts += (model.coef_ != 0)
      


    selection_prob = selection_counts / n_subsamples

    # Apply thresholds
    results[frac] = {
        thr: np.where(selection_prob >= thr)[0]
        for thr in thresholds
    }

for frac in results:
    print(f"\n=== Subsample fraction {frac} ===")
    for thr, feats in results[frac].items():
        print(f"  Threshold {thr}: {len(feats)} features selected")

  Progress: 0/100
  Progress: 10/100
  Progress: 20/100
  Progress: 30/100
  Progress: 40/100
  Progress: 50/100
  Progress: 60/100
  Progress: 70/100
  Progress: 80/100
  Progress: 90/100
  Progress: 0/100
  Progress: 10/100
  Progress: 20/100
  Progress: 30/100
  Progress: 40/100
  Progress: 50/100
  Progress: 60/100
  Progress: 70/100
  Progress: 80/100
  Progress: 90/100
  Progress: 0/100
  Progress: 10/100
  Progress: 20/100
  Progress: 30/100
  Progress: 40/100
  Progress: 50/100
  Progress: 60/100
  Progress: 70/100
  Progress: 80/100
  Progress: 90/100
  Progress: 0/100
  Progress: 10/100
  Progress: 20/100
  Progress: 30/100
  Progress: 40/100
  Progress: 50/100
  Progress: 60/100
  Progress: 70/100
  Progress: 80/100
  Progress: 90/100

=== Subsample fraction 0.3 ===
  Threshold 0.3: 161 features selected
  Threshold 0.5: 155 features selected
  Threshold 0.7: 120 features selected
  Threshold 0.9: 56 features selected

=== Subsample fraction 0.5 ===
  Threshold 0.3: 161 feat

In [ ]:
#Save the results so I don't have to run the stability selection again
import pickle

with open("stability_results.pkl", "wb") as f:
    pickle.dump(results, f)

In [ ]:

save_obj = {
    "results": results,
    "selection_prob": selection_prob,
    "feature_names": list(X_train_scaled.columns)
}

with open("stability_full_output.pkl", "wb") as f:
    pickle.dump(save_obj, f)


In [260]:
with open("stability_full_output.pkl", "rb") as f:
    data = pickle.load(f)

results = data["results"]
selection_prob = data["selection_prob"]
feature_names = data["feature_names"]

Trying RFECV - Recursive Feature Selection with Cross Validation to determine number of selected features

In [170]:
rfecv = RFECV(
    estimator=LogisticRegression(),
    step=1,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    verbose=2
)

rfecv.fit(X_train_scaled, y_train)

Fitting estimator with 163 features.
Fitting estimator with 162 features.
Fitting estimator with 161 features.
Fitting estimator with 160 features.
Fitting estimator with 159 features.
Fitting estimator with 158 features.
Fitting estimator with 157 features.
Fitting estimator with 156 features.
Fitting estimator with 155 features.
Fitting estimator with 154 features.
Fitting estimator with 153 features.
Fitting estimator with 152 features.
Fitting estimator with 151 features.
Fitting estimator with 150 features.
Fitting estimator with 149 features.
Fitting estimator with 148 features.
Fitting estimator with 147 features.
Fitting estimator with 146 features.
Fitting estimator with 145 features.
Fitting estimator with 144 features.
Fitting estimator with 143 features.
Fitting estimator with 142 features.
Fitting estimator with 141 features.
Fitting estimator with 140 features.
Fitting estimator with 139 features.
Fitting estimator with 138 features.
Fitting estimator with 137 features.
F

,estimator estimator: ``Estimator`` instanceA supervised learning estimator with a ``fit`` method that providesinformation about feature importance either through a ``coef_``attribute or through a ``feature_importances_`` attribute.,LogisticRegression()
,"step step: int or float, default=1If greater than or equal to 1, then ``step`` corresponds to the(integer) number of features to remove at each iteration.If within (0.0, 1.0), then ``step`` corresponds to the percentage(rounded down) of features to remove at each iteration.Note that the last iteration may remove fewer than ``step`` features inorder to reach ``min_features_to_select``.",1
,"min_features_to_select min_features_to_select: int, default=1The minimum number of features to be selected. This number of featureswill always be scored, even if the difference between the originalfeature count and ``min_features_to_select`` isn't divisible by``step``... versionadded:: 0.20",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If theestimator is not a classifier or if ``y`` is neither binary nor multiclass,:class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value of None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"scoring scoring: str or callable, default=NoneScoring method to evaluate the :class:`RFE` selectors' performance. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: the `estimator`'s :ref:`default evaluation criterion ` is used.",'roc_auc'
,"verbose verbose: int, default=0Controls verbosity of output.",2
,"n_jobs n_jobs: int or None, default=NoneNumber of cores to run in parallel while fitting across folds.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionadded:: 0.18",None
,"importance_getter importance_getter: str or callable, default='auto'If 'auto', uses the feature importance either through a `coef_`or `feature_importances_` attributes of estimator.Also accepts a string that specifies an attribute name/pathfor extracting feature importance.For example, give `regressor_.coef_` in case of:class:`~sklearn.compose.TransformedTargetRegressor` or`named_steps.clf.feature_importances_` in case of:class:`~sklearn.pipeline.Pipeline` with its last step named `clf`.If `callable`, overrides the default feature importance getter.The callable is passed with the fitted estimator and it shouldreturn importance for each feature... versionadded:: 0.24",'auto'
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a posit

Applying Models to Test data for Final Evaluations

In [ ]:
#Getting results for stability selection, including confusion matrix for 30% selection portion, 90% threshold combination

results = data['results']

metrics = []

thresholds = [0.3, 0.5, 0.7, 0.9]

#Going to make four confusion matrices for the 56 variable model 
fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(14,6), constrained_layout=True)


for i in results.keys():
    for j in results[i].keys():
        indexes = results[i][j]

        X_train_sub = X_train_scaled.iloc[:, indexes]
        X_test_sub = X_test_scaled.iloc[:, indexes]

        log_model = LogisticRegression(n_jobs=-1, verbose=1)
        log_model.fit(X_train_sub, y_train)

        y_prob = log_model.predict_proba(X_test_sub)[:, 1]

        for t in thresholds:
            y_pred = (y_prob >= t).astype(int)

            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall = recall_score(y_test, y_pred, zero_division=0)
            auc = roc_auc_score(y_test, y_prob) 

            if (i == .3) & (j== .9):
                conf_mat = confusion_matrix(y_pred=y_pred, y_true=y_test)   
                display = ConfusionMatrixDisplay(confusion_matrix = conf_mat, display_labels= ['Not Exciting', "Exciting"])
                if t == .3:
                    display.plot(ax=ax1, colorbar= False,cmap="Blues")
                    ax1.set_title("Threshold .3")
                elif t == .5:
                    display.plot(ax=ax2, colorbar= False,cmap="Blues")
                    ax2.set_title("Threshold .5")
                    ax2.set_ylabel("") 
                    ax2.set_yticklabels([])
                elif t == .7:
                    display.plot(ax=ax3, colorbar= False,cmap="Blues")
                    ax3.set_title("Threshold .7")
                    ax3.set_ylabel("") 
                    ax3.set_yticklabels([])
                elif t == .9:
                    display.plot(ax=ax4, cmap="Blues")
                    ax4.set_title("Threshold .9")
                    ax4.set_ylabel("") 
                    ax4.set_yticklabels([])

            metrics.append({
                "Subsample Proportion": i,
                "Selection Probability Threshold": j,
                "Log Reg Threshold": t,
                "numfeat": len(indexes),
                "indexes": indexes,
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "auc": auc
            })

        print(i, j, "Complete")

plt.savefig("Confusion Matrices")


C:\Users\LeBlancSamantha\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


0.3 0.3 Complete


C:\Users\LeBlancSamantha\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


0.3 0.5 Complete


C:\Users\LeBlancSamantha\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


...

0.9 0.5 Complete


C:\Users\LeBlancSamantha\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


0.9 0.7 Complete


C:\Users\LeBlancSamantha\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


0.9 0.9 Complete


In [ ]:
#Export data to html
metrics_df = pd.DataFrame(metrics)
metrics_df.drop(columns = ['indexes'], inplace = True)
metrics_df.to_html("metrics_table.html", index=False)
metrics_df.to_excel("metrics_df.xlsx")

In [ ]:
#Running a Logistic regression model on the 56 variable model and get coefficient and p-values

indexes = results[.3][.9]
X_train_sub = X_train_scaled.iloc[:, indexes]

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_sub, y_train)

log_coefs = log_model.coef_[0]
intercept = log_model.intercept_[0]

#Getting log modep probabilities for weight matrix
probs = log_model.predict_proba(X_train_sub)[:,1]

Weigh_mat = np.diag(probs * (1 -probs))


X_design = np.hstack([np.ones((X_train_sub.shape[0],1)), X_train_sub])

#Getting covariance matrix and standard errors
cov_matrix = np.linalg.inv(X_design.T @ Weigh_mat  @ X_design)

stand_errors = np.sqrt(np.diag(cov_matrix))


z_scores = np.concatenate([[intercept], log_coefs]) / stand_errors 
p_values = 2 * (1 - norm.cdf(np.abs(z_scores)))

summary_df = pd.DataFrame({
    "Feature": ["Intercept"] + list(X_train_sub.columns),
    "Coefficient": np.concatenate([[intercept], log_coefs]),
    "StdErr": stand_errors ,
    "z_score": z_scores,
    "p_value": p_values
})



In [ ]:
summary_df['abs_coef'] = abs(summary_df['Coefficient'])

In [ ]:

summary_df10 = summary_df.sort_values(by =["abs_coef", "p_value"], ascending = [False, True]).loc[summary_df['Feature'] != "Intercept"].iloc[0:10].drop(columns=['abs_coef'])
summary_df10.to_html("styled_table.html", index=False)

In [ ]:
summary_df.drop(columns=['abs_coef']).to_html("styled_table_full.html", index=False)

In [179]:
#Full Model - RFE

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)

coefs = log_model.coef_[0]
intercept = log_model.intercept_[0]

# Predicted probabilities
p = log_model.predict_proba(X_train_scaled)[:,1]

# Weight matrix
W = np.diag(p * (1 - p))

# Add intercept if needed
X_design = np.hstack([np.ones((X_train_scaled.shape[0],1)), X_train_scaled])

# Covariance matrix
cov_matrix = np.linalg.inv(X_design.T @ W @ X_design)

# Standard errors
se = np.sqrt(np.diag(cov_matrix))

from scipy.stats import norm

z_scores = np.concatenate([[intercept], coefs]) / se
p_values = 2 * (1 - norm.cdf(np.abs(z_scores)))

summary_df_rfe = pd.DataFrame({
    "Feature": ["Intercept"] + list(X_train_scaled.columns),
    "Coefficient": np.concatenate([[intercept], coefs]),
    "StdErr": se,
    "z_score": z_scores,
    "p_value": p_values
})

#summary_df_rfe.sort_values(by =["Coefficient", "p_value"], ascending=[False, True])

C:\Users\LeBlancSamantha\AppData\Local\Temp\ipykernel_38928\1121194868.py:22: RuntimeWarning: invalid value encountered in sqrt
  se = np.sqrt(np.diag(cov_matrix))


In [180]:
summary_df_rfe['abs_coef'] = abs(summary_df_rfe['Coefficient'])
summary_df10_rfe = summary_df_rfe.loc[(summary_df_rfe['p_value'] <= .05) & (summary_df_rfe['Feature'] != "Intercept")].sort_values(by =["abs_coef", "p_value"], ascending = [False, True]).iloc[0:10].drop(columns=['abs_coef'])
summary_df10_rfe.to_html("styled_table_rfe.html", index=False)
summary_df_rfe.drop(columns=['abs_coef']).to_html("styled_table_rfe_full.html", index=False)

In [185]:
# Calculated RFE Metrics 

rfe_metrics = []

thresholds = [0.3, 0.5, 0.7, 0.9]


log_model = LogisticRegression(n_jobs=-1, verbose=1)
log_model.fit(X_train_scaled, y_train)

y_prob = log_model.predict_proba(X_test_scaled)[:, 1]

for t in thresholds:
    y_pred = (y_prob >= t).astype(int)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)  # same across thresholds

    rfe_metrics.append({
                "Log Reg Threshold": t,
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "auc": auc
            })
    
rfe_metrics_df = pd.DataFrame(rfe_metrics)
rfe_metrics_df.to_excel("rfe_metrics.xlsx")



C:\Users\LeBlancSamantha\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


In [ ]:
#Grouped bar chart to plot metrics for large vs. small logistic regression model
df56 = metrics_df[metrics_df['numfeat'] == 56]
df162 = metrics_df[metrics_df['numfeat'] == 162]

def plot_grouped(df, title, ax):
    df = df.sort_values('Log Reg Threshold')

    thresholds = df['Log Reg Threshold'].values
    x = np.arange(len(thresholds))

    width = 0.2

    ax.bar(x - 1.5*width, df['accuracy'], width, label='Accuracy', color = "cornflowerblue")
    ax.bar(x - 0.5*width, df['precision'], width, label='Precision', color = "lightcoral")
    ax.bar(x + 0.5*width, df['recall'], width, label='Recall', color = "mediumseagreen")
    ax.bar(x + 1.5*width, df['auc'], width, label='AUC', color = "orange")

    ax.set_xticks(x)
    ax.set_xticklabels(thresholds)
    ax.set_xlabel("Logistic Regression Probability Threshold")
    ax.set_ylabel("Score")
    ax.set_title(title)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,6))
plot_grouped(df56, "Logistic Model - 56 Features", ax1)
plot_grouped(df162, "Logistic Model - 162 features", ax2)
handles, labels = ax1.get_legend_handles_labels()

fig.legend(handles, labels, loc='upper right')

plt.tight_layout()
plt.savefig("grouped_chart")

In [ ]:
#Doing same for RFE
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,6))
plot_grouped(df56, "Logistic Model - 56 Features", ax1)
plot_grouped(rfe_metrics_df, "RFE Full Model - 163 Features", ax2)
handles, labels = ax1.get_legend_handles_labels()

fig.legend(handles, labels, loc='upper right')

plt.tight_layout()
plt.savefig("grouped_chart_rfe")

C:\Users\LeBlancSamantha\AppData\Local\Temp\ipykernel_38928\1927871254.py:1: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,6))
